# Laboratorio 27 — QML con datos reales: clasificador cuántico vs. SVM clásico

Clasificamos el dataset Iris (reducido a 2 clases, 2 características) con:
1. **SVM clásico** con kernel RBF — referencia.
2. **Kernel cuántico** (quantum feature map) con `StatevectorEstimator`.
3. **Circuito variacional (VQC)** optimizado con gradiente.

Comparamos: accuracy, tiempo de entrenamiento, expresividad del ansatz.

**Prerrequisitos:** Módulos 12 (QML), 11 (VQE/QAOA), scikit-learn instalado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

print('Imports OK — scikit-learn + Qiskit')

## 1. Preparación del dataset Iris (binario)

Usamos solo Iris-setosa vs. Iris-versicolor (2 clases) y las 2 primeras características
(longitud y anchura del sépalo). Esto permite visualizar la frontera de decisión en 2D.

In [ ]:
iris = load_iris()

# Tomar 2 clases (0=setosa, 1=versicolor) y 2 características
mask = iris.target < 2
X = iris.data[mask, :2]   # longitud sépalo, anchura sépalo
y = iris.target[mask]     # etiquetas 0, 1

# Normalizar a [0, π] para usarlas como ángulos de rotación
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Dataset: {X.shape[0]} muestras, 2 características')
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')
print(f'Clases: setosa (0) = {sum(y==0)}, versicolor (1) = {sum(y==1)}')

# Visualizar
fig, ax = plt.subplots(figsize=(7, 5))
for cls, lbl, color in [(0, 'Setosa', '#3498db'), (1, 'Versicolor', '#e74c3c')]:
    mask_c = y == cls
    ax.scatter(X_scaled[mask_c, 0], X_scaled[mask_c, 1],
               color=color, label=lbl, s=60, alpha=0.8)
ax.set_xlabel('Longitud sépalo (normalizada 0-π)')
ax.set_ylabel('Anchura sépalo (normalizada 0-π)')
ax.set_title('Iris: Setosa vs. Versicolor')
ax.legend()
plt.show()

## 2. Referencia clásica: SVM con kernel RBF

In [ ]:
t0 = time.time()
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train, y_train)
t_svm = time.time() - t0

y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'SVM (kernel RBF):')
print(f'  Accuracy test:    {acc_svm*100:.1f}%')
print(f'  Tiempo de train:  {t_svm*1000:.1f} ms')

## 3. Kernel cuántico con ZZFeatureMap

El kernel cuántico calcula la similitud entre dos puntos como el overlap de sus estados
cuánticos: $k(x, x') = |\langle \phi(x) | \phi(x') \rangle|^2$

Usamos el mapa de características ZZ (Havlíček et al., 2019).

In [ ]:
def zz_feature_map(x: np.ndarray, reps: int = 2) -> QuantumCircuit:
    """ZZFeatureMap: codifica x en amplitudes cuánticas con entrelazamiento."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        # Capa de Hadamard
        qc.h(range(n))
        # Rotaciones individuales
        for i in range(n):
            qc.p(2 * x[i], i)   # P = Rz salvo fase global
        # Entrelazamiento: rotaciones cruzadas
        for i in range(n):
            for j in range(i+1, n):
                qc.cx(i, j)
                qc.p(2 * (np.pi - x[i]) * (np.pi - x[j]), j)
                qc.cx(i, j)
    return qc

def kernel_cuantico(x1: np.ndarray, x2: np.ndarray) -> float:
    """Kernel cuántico: |⟨φ(x1)|φ(x2)⟩|²"""
    qc1 = zz_feature_map(x1)
    qc2 = zz_feature_map(x2)
    sv1 = Statevector.from_instruction(qc1)
    sv2 = Statevector.from_instruction(qc2)
    return float(abs(sv1.inner(sv2))**2)

# Construir matriz del kernel cuántico
def gram_matrix(X_a, X_b):
    n_a, n_b = len(X_a), len(X_b)
    K = np.zeros((n_a, n_b))
    for i in range(n_a):
        for j in range(n_b):
            K[i, j] = kernel_cuantico(X_a[i], X_b[j])
    return K

print('Construyendo matrices del kernel cuántico (puede tardar ~30s)...')
t0 = time.time()
K_train = gram_matrix(X_train, X_train)
K_test  = gram_matrix(X_test, X_train)
t_kernel = time.time() - t0
print(f'Matrices kernel construidas en {t_kernel:.1f}s')

# SVM con kernel cuántico precomputed
svm_q = SVC(kernel='precomputed', C=1.0)
svm_q.fit(K_train, y_train)
y_pred_qk = svm_q.predict(K_test)
acc_qk = accuracy_score(y_test, y_pred_qk)
print(f'\nSVM + Kernel Cuántico (ZZFeatureMap):')
print(f'  Accuracy test:    {acc_qk*100:.1f}%')
print(f'  Tiempo de kernel: {t_kernel:.1f}s')

## 4. Circuito variacional (VQC) optimizado

In [ ]:
def build_vqc(x: np.ndarray, theta: np.ndarray) -> QuantumCircuit:
    """VQC: ZZFeatureMap + ansatz variacional RealAmplitudes."""
    n = len(x)
    qc = QuantumCircuit(n)
    # Codificación de datos
    qc.compose(zz_feature_map(x, reps=1), inplace=True)
    # Ansatz variacional (2 capas, n parámetros por capa)
    n_params_per_layer = n
    for layer in range(2):
        offset = layer * n_params_per_layer
        for i in range(n):
            qc.ry(theta[offset + i], i)
        if layer < 1:
            for i in range(n-1):
                qc.cx(i, i+1)
    return qc

def predict_vqc(x: np.ndarray, theta: np.ndarray) -> float:
    """Devuelve ⟨Z₀⟩ como salida del clasificador."""
    qc = build_vqc(x, theta)
    sv = Statevector.from_instruction(qc)
    Z = SparsePauliOp('IZ')  # Z en qubit 0 (Qiskit: qubit 0 es el último bit)
    return float(sv.expectation_value(Z).real)

def vqc_loss(theta: np.ndarray, X_data, y_data) -> float:
    """MSE sobre salidas del VQC (etiquetas: -1 = clase 0, +1 = clase 1)."""
    y_scaled = 2 * y_data - 1   # {0,1} → {-1,+1}
    preds = np.array([predict_vqc(x, theta) for x in X_data])
    return float(np.mean((preds - y_scaled)**2))

# Entrenar VQC
n_features = X_train.shape[1]
n_params = 2 * n_features   # 2 capas × n_features params
np.random.seed(42)
theta0 = np.random.uniform(0, np.pi, n_params)

print('Entrenando VQC (puede tardar ~1-2 min)...')
t0 = time.time()
res_vqc = minimize(
    vqc_loss, theta0,
    args=(X_train, y_train),
    method='COBYLA',
    options={'maxiter': 200}
)
t_vqc = time.time() - t0

# Predicción VQC
theta_opt = res_vqc.x
preds_raw = np.array([predict_vqc(x, theta_opt) for x in X_test])
y_pred_vqc = (preds_raw > 0).astype(int)   # umbral en 0
acc_vqc = accuracy_score(y_test, y_pred_vqc)

print(f'\nVQC (ZZFeatureMap + RealAmplitudes 2 capas):')
print(f'  Accuracy test:    {acc_vqc*100:.1f}%')
print(f'  Tiempo de train:  {t_vqc:.1f}s')
print(f'  Loss final:       {res_vqc.fun:.4f}')

## 5. Comparativa final

In [ ]:
print('\n' + '='*55)
print('COMPARATIVA FINAL — Iris binario (n=100, d=2)')
print('='*55)
print(f"{'Modelo':<30} {'Accuracy':>10} {'Tiempo':>12}")
print('-' * 55)
print(f"{'SVM (kernel RBF, clásico)':<30} {acc_svm*100:>9.1f}% {t_svm*1000:>10.0f}ms")
print(f"{'SVM + Kernel Cuántico':<30} {acc_qk*100:>9.1f}% {t_kernel:>10.1f}s")
print(f"{'VQC variacional':<30} {acc_vqc*100:>9.1f}% {t_vqc:>10.1f}s")

# Gráfica de accuracy
modelos = ['SVM\nRBF', 'Kernel\nCuántico', 'VQC\nVariacional']
accuracies = [acc_svm*100, acc_qk*100, acc_vqc*100]
colors = ['#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(modelos, accuracies, color=colors, alpha=0.85, width=0.5)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparativa de clasificadores: Iris setosa vs. versicolor')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.axhline(y=100, color='gray', linestyle=':', alpha=0.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Análisis: ¿el clasificador cuántico tiene ventaja?

Para el dataset Iris binario (casi linealmente separable), el SVM clásico es difícil
de superar. Los clasificadores cuánticos tienen el mismo accuracy o inferior,
y tardan varios órdenes de magnitud más.

**¿Cuándo puede haber ventaja cuántica en QML?**

1. **Datos con estructura cuántica:** resultados de experimentos cuánticos donde
   el feature map cuántico es más natural que el clásico.
2. **Espacios de características exponencialmente grandes:** el kernel cuántico vive
   en un espacio de Hilbert de dimensión 2^n, inaccesible al kernel clásico.
3. **Clasificación de fases cuánticas:** el VQC puede aprender a distinguir fases
   topológicas donde los classificadores clásicos necesitan más muestras.

**Barren plateaus:** el gradiente del VQC decae exponencialmente con el número de
qubits para ansatz aleatorios. Esto hace que el entrenamiento sea exponencialmente
difícil para circuitos profundos — una limitación fundamental del QML actual.

In [ ]:
# Visualizar frontera de decisión del VQC
xx, yy = np.meshgrid(np.linspace(0, np.pi, 30), np.linspace(0, np.pi, 30))
grid = np.c_[xx.ravel(), yy.ravel()]

preds_grid = np.array([predict_vqc(pt, theta_opt) for pt in grid])
Z_grid = preds_grid.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (titulo, Z_plot, y_pred) in zip(axes, [
    ('SVM kernel RBF', svm.decision_function(grid).reshape(xx.shape), y_pred_svm),
    ('VQC cuántico', Z_grid, y_pred_vqc)
]):
    ax.contourf(xx, yy, Z_plot, alpha=0.3, cmap='RdBu', levels=20)
    for cls, lbl, color in [(0, 'Setosa', '#3498db'), (1, 'Versicolor', '#e74c3c')]:
        mask_c = y_test == cls
        # Correctos
        corr = y_test == (y_pred_svm if titulo.startswith('SVM') else y_pred_vqc)
        ax.scatter(X_test[mask_c & corr, 0], X_test[mask_c & corr, 1],
                   color=color, s=60, marker='o', label=f'{lbl} (correcto)')
        ax.scatter(X_test[mask_c & ~corr, 0], X_test[mask_c & ~corr, 1],
                   color=color, s=80, marker='x', label=f'{lbl} (error)')
    ax.set_title(titulo)
    ax.set_xlabel('Longitud sépalo')
    ax.set_ylabel('Anchura sépalo')
    ax.legend(fontsize=7, loc='lower right')

plt.suptitle('Fronteras de decisión: SVM clásico vs. VQC cuántico')
plt.tight_layout()
plt.show()

print('\nMódulos relacionados: 12_aplicaciones (QML), 22_recursos_cuanticos (expresividad)')